# Predictive CLTV Dataset Preparation

## Overview

The objective of this stage is to prepare a modelling-ready dataset for predictive Customer Lifetime Value (CLTV) analysis.

The customer-level dataset developed during Week 3 already contains several behavioural and transactional metrics. During this stage, these variables are validated and additional modelling features are engineered to improve the predictive capability of the dataset.

The resulting dataset will serve as the input for predictive CLTV modelling in the next stage of the project.

In [1]:
# ==========================================
# Import Libraries
# ==========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==========================================
# Project Paths
# ==========================================

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
IMAGES = PROJECT_ROOT / "images"
REPORTS = PROJECT_ROOT / "reports"

print("Project directories configured successfully.")

Project directories configured successfully.


In [3]:
import os

print(sorted(os.listdir(PROCESSED_DATA)))

['.gitkeep', 'cohort_counts.csv', 'cohort_sizes.csv', 'customer_segments.csv', 'customer_summary.csv', 'executive_summary.csv', 'online_retail_clean.csv', 'retention_matrix.csv', 'segment_metrics.csv', 'top10_cltv_customers.csv']


In [4]:
# ==========================================
# Load Customer Summary Dataset
# ==========================================

customer_summary = pd.read_csv(
    PROCESSED_DATA / "customer_summary.csv",
    parse_dates=["FirstPurchase", "LastPurchase"]
)

print("Customer Summary Dataset Loaded Successfully")
print(customer_summary.shape)

customer_summary.head()

Customer Summary Dataset Loaded Successfully
(5878, 12)


,Customer ID,TotalRevenue,TotalOrders,TotalTransactions,TotalProducts,FirstPurchase,LastPurchase,CustomerLifespan,AverageOrderValue,PurchaseFrequency,CustomerAge,HistoricalCLTV
0,12346.0,77556.46,12,34,74285,2009-12-14 08:34:00,2011-01-18 10:01:00,400,6463.038333,6.289384,725,16259412.33
1,12347.0,4921.53,8,222,2967,2010-10-31 14:20:00,2011-12-07 15:52:00,402,615.191250,6.289384,403,1555407.99
2,12348.0,2019.40,5,51,2714,2010-09-27 14:59:00,2011-09-25 13:13:00,362,403.880000,6.289384,437,919536.64
3,12349.0,4428.69,4,175,1624,2010-04-29 13:20:00,2011-11-21 09:51:00,570,1107.172500,6.289384,588,3969156.90
4,12350.0,334.40,1,17,197,2011-02-02 16:01:00,2011-02-02 16:01:00,0,334.400000,6.289384,309,0.00


In [5]:
# ==========================================
# Dataset Validation
# ==========================================

print("Dataset Shape:", customer_summary.shape)

print("\nColumn Names:")
print(customer_summary.columns.tolist())

print("\nMissing Values:")
print(customer_summary.isnull().sum())

print("\nDataset Information:")
customer_summary.info()

Dataset Shape: (5878, 12)

Column Names:
['Customer ID', 'TotalRevenue', 'TotalOrders', 'TotalTransactions', 'TotalProducts', 'FirstPurchase', 'LastPurchase', 'CustomerLifespan', 'AverageOrderValue', 'PurchaseFrequency', 'CustomerAge', 'HistoricalCLTV']

Missing Values:
Customer ID          0
TotalRevenue         0
TotalOrders          0
TotalTransactions    0
TotalProducts        0
FirstPurchase        0
LastPurchase         0
CustomerLifespan     0
AverageOrderValue    0
PurchaseFrequency    0
CustomerAge          0
HistoricalCLTV       0
dtype: int64

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 5878 entries, 0 to 5877
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Customer ID        5878 non-null   float64       
 1   TotalRevenue       5878 non-null   float64       
 2   TotalOrders        5878 non-null   int64         
 3   TotalTransactions  5878 non-null   int64  

In [6]:
# ==========================================
# Feature Engineering: Customer Recency
# ==========================================

reference_date = customer_summary["LastPurchase"].max()

customer_summary["CustomerRecency"] = (
    reference_date - customer_summary["LastPurchase"]
).dt.days

customer_summary[
    ["LastPurchase", "CustomerRecency"]
].head()

,LastPurchase,CustomerRecency
0,2011-01-18 10:01:00,325
1,2011-12-07 15:52:00,1
2,2011-09-25 13:13:00,74
3,2011-11-21 09:51:00,18
4,2011-02-02 16:01:00,309


In [7]:
# ==========================================
# Feature Engineering: Revenue per Product
# ==========================================

customer_summary["RevenuePerProduct"] = (
    customer_summary["TotalRevenue"] /
    customer_summary["TotalProducts"]
)

customer_summary[
    ["TotalRevenue", "TotalProducts", "RevenuePerProduct"]
].head()

,TotalRevenue,TotalProducts,RevenuePerProduct
0,77556.46,74285,1.044039
1,4921.53,2967,1.658756
2,2019.40,2714,0.744068
3,4428.69,1624,2.727026
4,334.40,197,1.697462


In [8]:
# ==========================================
# Feature Engineering: Revenue per Transaction
# ==========================================

customer_summary["RevenuePerTransaction"] = (
    customer_summary["TotalRevenue"] /
    customer_summary["TotalTransactions"]
)

customer_summary[
    ["TotalRevenue", "TotalTransactions", "RevenuePerTransaction"]
].head()

,TotalRevenue,TotalTransactions,RevenuePerTransaction
0,77556.46,34,2281.072353
1,4921.53,222,22.169054
2,2019.40,51,39.596078
3,4428.69,175,25.306800
4,334.40,17,19.670588


In [10]:
# ==========================================
# Feature Summary
# ==========================================

customer_summary[
    [
        "CustomerRecency",
        "RevenuePerProduct",
        "RevenuePerTransaction"
    ]
].describe().round(2)

,CustomerRecency,RevenuePerProduct,RevenuePerTransaction
count,5878.00,5878.00,5878.00
mean,200.33,7.15,48.30
std,209.34,176.23,780.18
min,0.00,0.12,2.14
25%,25.00,1.43,11.56
50%,95.00,1.83,17.37
75%,379.00,2.35,24.18
max,738.00,10953.50,56157.50


In [11]:
# ==========================================
# Save Predictive CLTV Dataset
# ==========================================

customer_summary.to_csv(
    PROCESSED_DATA / "cltv_model_dataset.csv",
    index=False
)

print("Predictive CLTV dataset saved successfully.")
print(customer_summary.shape)

Predictive CLTV dataset saved successfully.
(5878, 15)


## Feature Engineering Summary

To prepare the dataset for predictive Customer Lifetime Value (CLTV) modelling, additional behavioural features were engineered and validated.

### New Features

- **Customer Recency:** Number of days since the customer's most recent purchase.
- **Revenue per Product:** Average revenue generated per product purchased.
- **Revenue per Transaction:** Average revenue generated per transaction.

The engineered features enhance the modelling dataset by incorporating customer engagement and spending behaviour, creating a richer feature set for predictive analytics. The completed modelling dataset was exported for use in the predictive CLTV modelling stage.